# 06 — Audit sumber, koreksi anatomi, dan preprocessing 2.000 Hz

Jalankan sel berurutan. Notebook ini tidak melatih model. Hasil lama dipertahankan.

Sumber: [paper 2022](https://pmc.ncbi.nlm.nih.gov/articles/PMC8962834/) dan [descriptor dataset 334 pasien](https://www.nature.com/articles/s41597-020-0440-8).

Tiga kelompok disetujui pengguna: **LVOT**, **RVOT cusp (LC/AC/RC)**, **RVOT septal/free-wall/other**. Ini kelompok anatomi khusus penelitian, bukan LVOT/Non-OT/RVOT. Label numerik dan 289 pasien tetap sama. Summit/unknown tetap dikecualikan untuk menjaga kohort; bukan pernyataan bahwa semua Summit tidak termasuk LVOT.

Sampling publik 2.000 Hz cocok dengan asal/struktur lokal; checksum unduhan asli belum dicocokkan. 5.000 sampel = 2,5 detik. Segmen belum dipusatkan pada PVC teranotasi. Preprocessing baru adalah kandidat terkontrol, belum terbukti meningkatkan kinerja.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "research_v2").is_dir() else Path.cwd().parent
assert (ROOT / "research_v2").is_dir(), "Buka notebook dari folder project/notebooks"
sys.path.insert(0, str(ROOT / "notebooks"))
from _research_runner import ResearchRunner
RUN_ID = "20260930_anatomy_fs2000_v2"
DEVICE = "mps"  # Mac Apple Silicon; ubah ke "cpu" bila perlu.
runner = ResearchRunner(ROOT, RUN_ID, DEVICE, label="v2_manual")
RUN = ROOT / "results/research_v2" / RUN_ID


In [ ]:
runner.run_stage("prepare")
runner.run_stage("verify")
runner.run_stage("audit")

In [ ]:
display(pd.read_csv(RUN / "reports/label_mapping_counts.csv"))
display(Image(filename=str(RUN / "reports/cohort_groups.png")))
display(Image(filename=str(RUN / "reports/preprocessing_review.png")))

## Preprocessing development

Sel berikut memproses 230 pasien development dari raw dengan HPF SOS 0,5 Hz pada 2.000 Hz + coif5 level 5 SURE. Tidak membaca sinyal test. Tidak melakukan resampling. Output memory map baru; sediakan ruang disk. Periksa morfologi bersama ahli ECG; grafik tiga contoh bukan validasi klinis lengkap.

Segmen, overlap, fold pasien, resep classifier, dan sembilan LR×batch dipertahankan. Perbandingan antar-dua arm V2 mengisolasi class weighting. Perbandingan V2 versus hasil lama sekaligus mengubah preprocessing dan augmentasi, sehingga bukan estimasi murni efek GAN.

In [ ]:
runner.run_stage("preprocess")

In [ ]:
import json
manifest = json.loads((RUN / "data/development/manifest.json").read_text())
print({k: manifest[k] for k in ["n_patients", "n_segments", "sampling_rate"]})

Lanjutkan notebook 07. Tidak perlu menjalankan ulang notebook 00–05.